# Production RAG: Observability with MLflow

## Learning Objectives
- Set up MLflow for experiment tracking
- Monitor RAG system performance
- Create custom visualizations
- Implement automated quality checks
- Build production-ready RAG with monitoring

## Prerequisites
- Completed Sessions 1 & 2
- MLflow installed and configured

In [ ]:
import os
import mlflow
import mlflow.sklearn
import google.generativeai as genai
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

print("Session 3: Production RAG with MLflow Observability")

## 1. MLflow Setup

MLflow tracks experiments, logs metrics, and manages models.

### Why MLflow for RAG?
- Track different chunking strategies
- Compare retrieval methods
- Log generation quality scores
- Version your RAG pipelines

In [ ]:
# Set up MLflow tracking
mlflow.set_tracking_uri("sqlite:///tmp/mlflow.db")  # SQLite backend for demo
mlflow.set_experiment("rag_production_demo")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("MLflow experiment created!")

In [ ]:
class MonitoredRAG:
    """
    RAG system with MLflow monitoring.
    """
    
    def __init__(self):
        self.query_log = []
        self.retrieval_metrics = []
        self.generation_metrics = []
        
    def log_query(self, query, results, answer, latency_metrics):
        """Log a query and its results."""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'query': query,
            'num_results': len(results),
            'answer_length': len(answer),
            'retrieval_latency': latency_metrics.get('retrieval_latency', 0),
            'generation_latency': latency_metrics.get('generation_latency', 0),
            'total_latency': latency_metrics.get('total_latency', 0),
            'avg_retrieval_score': sum(r.get('score', 0) for r in results) / len(results) if results else 0
        }
        self.query_log.append(log_entry)
        return log_entry
    
    def get_statistics(self):
        """Get aggregate statistics."""
        if not self.query_log:
            return {}
        
        df = pd.DataFrame(self.query_log)
        return {
            'total_queries': len(df),
            'avg_retrieval_latency': df['retrieval_latency'].mean(),
            'avg_generation_latency': df['generation_latency'].mean(),
            'avg_total_latency': df['total_latency'].mean(),
            'avg_retrieval_score': df['avg_retrieval_score'].mean()
        }


# Create monitored RAG instance
monitored_rag = MonitoredRAG()
print("Monitored RAG system initialized!")

## 2. Experiment Tracking

Let's run experiments with different configurations and track them in MLflow.

In [ ]:
def run_rag_experiment(
    experiment_name,
    chunk_method,
    chunk_size,
    search_method,
    test_queries
):
    """
    Run a complete RAG experiment with MLflow tracking.
    """
    with mlflow.start_run(run_name=experiment_name):
        # Log parameters
        mlflow.log_param("chunk_method", chunk_method)
        mlflow.log_param("chunk_size", chunk_size)
        mlflow.log_param("search_method", search_method)
        mlflow.log_param("num_test_queries", len(test_queries))
        
        # Initialize metrics
        retrieval_latencies = []
        generation_latencies = []
        retrieval_scores = []
        
        # Run queries
        for query in test_queries:
            start_time = datetime.now()
            
            # Simulate retrieval (in real scenario, use actual RAG)
            # This is a simplified example
            retrieval_time = 0.1  # Simulated
            
            # Simulate generation
            generation_time = 0.5  # Simulated
            
            total_time = retrieval_time + generation_time
            
            # Log metrics for each query
            mlflow.log_metric("retrieval_latency", retrieval_time)
            mlflow.log_metric("generation_latency", generation_time)
            mlflow.log_metric("total_latency", total_time)
            mlflow.log_metric("retrieval_score", 0.85)  # Simulated
            
            retrieval_latencies.append(retrieval_time)
            generation_latencies.append(generation_time)
            retrieval_scores.append(0.85)
        
        # Log summary metrics
        mlflow.log_metric("avg_retrieval_latency", sum(retrieval_latencies) / len(retrieval_latencies))
        mlflow.log_metric("avg_generation_latency", sum(generation_latencies) / len(generation_latencies))
        mlflow.log_metric("avg_retrieval_score", sum(retrieval_scores) / len(retrieval_scores))
        
        # Log tags
        mlflow.set_tag("session", "production_observability")
        mlflow.set_tag("model", "gemini-1.5-pro")
        
        print(f"Experiment '{experiment_name}' completed!")
        print(f"  Average retrieval latency: {sum(retrieval_latencies) / len(retrieval_latencies):.3f}s")
        print(f"  Average generation latency: {sum(generation_latencies) / len(generation_latencies):.3f}s")
        print(f"  Average retrieval score: {sum(retrieval_scores) / len(retrieval_scores):.3f}")


# Run multiple experiments
test_queries = [
    "What is machine learning?",
    "How do neural networks work?",
    "Explain deep learning",
    "What is NLP?",
    "How does transformers work?"
]

print("Running experiments with different configurations...\n")

# Experiment 1: Small chunks
run_rag_experiment(
    experiment_name="small_chunks_recursive",
    chunk_method="recursive",
    chunk_size=200,
    search_method="semantic",
    test_queries=test_queries
)

# Experiment 2: Large chunks
run_rag_experiment(
    experiment_name="large_chunks_recursive",
    chunk_method="recursive",
    chunk_size=500,
    search_method="semantic",
    test_queries=test_queries
)

# Experiment 3: Hybrid search
run_rag_experiment(
    experiment_name="hybrid_search_rrf",
    chunk_method="recursive",
    chunk_size=300,
    search_method="hybrid_rrf",
    test_queries=test_queries
)

## 3. Custom Visualizations

Let's create visualizations to understand our RAG system's performance.

In [ ]:
def create_performance_dashboard():
    """Create a performance visualization dashboard."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Simulated data for demonstration
    experiments = ['Small Chunks', 'Large Chunks', 'Hybrid Search']
    retrieval_latencies = [0.08, 0.12, 0.15]
    generation_latencies = [0.45, 0.42, 0.48]
    retrieval_scores = [0.82, 0.78, 0.89]
    chunk_counts = [150, 75, 100]
    
    # Plot 1: Latency comparison
    x = range(len(experiments))
    width = 0.35
    
    axes[0, 0].bar(x, retrieval_latencies, width, label='Retrieval', color='blue')
    axes[0, 0].bar([i + width for i in x], generation_latencies, width, label='Generation', color='orange')
    axes[0, 0].set_xlabel('Experiment')
    axes[0, 0].set_ylabel('Latency (seconds)')
    axes[0, 0].set_title('Latency by Component')
    axes[0, 0].set_xticks([i + width/2 for i in x])
    axes[0, 0].set_xticklabels(experiments)
    axes[0, 0].legend()
    
    # Plot 2: Retrieval scores
    axes[0, 1].bar(experiments, retrieval_scores, color=['green', 'red', 'purple'])
    axes[0, 1].set_xlabel('Experiment')
    axes[0, 1].set_ylabel('Retrieval Score')
    axes[0, 1].set_title('Retrieval Quality Comparison')
    axes[0, 1].set_ylim(0, 1)
    
    # Plot 3: Chunk count vs performance
    axes[1, 0].scatter(chunk_counts, retrieval_scores, s=100, c=['blue', 'red', 'green'])
    axes[1, 0].set_xlabel('Number of Chunks')
    axes[1, 0].set_ylabel('Retrieval Score')
    axes[1, 0].set_title('Chunk Count vs Retrieval Quality')
    for i, txt in enumerate(experiments):
        axes[1, 0].annotate(txt, (chunk_counts[i], retrieval_scores[i]), 
                           xytext=(5, 5), textcoords='offset points')
    
    # Plot 4: Total latency breakdown
    total_latencies = [r + g for r, g in zip(retrieval_latencies, generation_latencies)]
    axes[1, 1].pie(total_latencies, labels=experiments, autopct='%1.1f%%',
                   colors=['lightblue', 'lightcoral', 'lightgreen'])
    axes[1, 1].set_title('Total Latency Distribution')
    
    plt.tight_layout()
    plt.show()
    
    print("Dashboard created!")


create_performance_dashboard()

In [ ]:
def create_retrieval_analysis_viz():
    """Visualize retrieval analysis."""
    # Simulated retrieval data
    queries = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
    semantic_scores = [0.85, 0.72, 0.91, 0.68, 0.79]
    bm25_scores = [0.78, 0.88, 0.75, 0.82, 0.71]
    hybrid_scores = [0.89, 0.85, 0.93, 0.81, 0.87]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Score comparison across queries
    x = range(len(queries))
    width = 0.25
    
    axes[0].bar(x, semantic_scores, width, label='Semantic', color='blue')
    axes[0].bar([i + width for i in x], bm25_scores, width, label='BM25', color='orange')
    axes[0].bar([i + 2*width for i in x], hybrid_scores, width, label='Hybrid', color='green')
    
    axes[0].set_xlabel('Query')
    axes[0].set_ylabel('Retrieval Score')
    axes[0].set_title('Retrieval Scores by Method')
    axes[0].set_xticks([i + width for i in x])
    axes[0].set_xticklabels(queries)
    axes[0].legend()
    axes[0].set_ylim(0, 1)
    
    # Plot 2: Method improvement
    improvements = [(h - s) / s * 100 for h, s in zip(hybrid_scores, semantic_scores)]
    axes[1].bar(queries, improvements, color='green')
    axes[1].set_xlabel('Query')
    axes[1].set_ylabel('Improvement (%)')
    axes[1].set_title('Hybrid vs Semantic Search Improvement')
    axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()


create_retrieval_analysis_viz()

## 4. Automated Quality Checks

Implement automated checks to monitor RAG quality in production.

In [ ]:
class RAGQualityMonitor:
    """
    Automated quality monitoring for RAG systems.
    """
    
    def __init__(self, thresholds=None):
        self.thresholds = thresholds or {
            'min_retrieval_score': 0.6,
            'max_latency': 2.0,
            'min_answer_length': 50
        }
        self.alerts = []
        
    def check_quality(self, query, results, answer, latency):
        """Run quality checks and return alerts."""
        checks_passed = True
        check_results = []
        
        # Check 1: Retrieval quality
        avg_score = sum(r.get('score', 0) for r in results) / len(results) if results else 0
        if avg_score < self.thresholds['min_retrieval_score']:
            self.alerts.append({
                'type': 'LOW_RETRIEVAL_SCORE',
                'severity': 'WARNING',
                'message': f'Low retrieval score: {avg_score:.3f}',
                'query': query
            })
            checks_passed = False
        check_results.append(('Retrieval Score', avg_score, avg_score >= self.thresholds['min_retrieval_score']))
        
        # Check 2: Latency
        if latency > self.thresholds['max_latency']:
            self.alerts.append({
                'type': 'HIGH_LATENCY',
                'severity': 'WARNING',
                'message': f'High latency: {latency:.3f}s',
                'query': query
            })
            checks_passed = False
        check_results.append(('Latency', latency, latency <= self.thresholds['max_latency']))
        
        # Check 3: Answer quality
        answer_length = len(answer) if answer else 0
        if answer_length < self.thresholds['min_answer_length']:
            self.alerts.append({
                'type': 'SHORT_ANSWER',
                'severity': 'INFO',
                'message': f'Short answer: {answer_length} chars',
                'query': query
            })
        check_results.append(('Answer Length', answer_length, answer_length >= self.thresholds['min_answer_length']))
        
        return {
            'checks_passed': checks_passed,
            'check_results': check_results,
            'alerts': self.alerts.copy()
        }
    
    def get_alert_summary(self):
        """Get summary of all alerts."""
        alert_counts = {}
        for alert in self.alerts:
            alert_type = alert['type']
            alert_counts[alert_type] = alert_counts.get(alert_type, 0) + 1
        return alert_counts


# Test quality monitoring
monitor = RAGQualityMonitor()

# Simulate queries with quality issues
test_scenarios = [
    {
        'query': 'What is ML?',
        'results': [{'score': 0.4}, {'score': 0.3}],  # Low scores
        'answer': 'Short answer',
        'latency': 0.5
    },
    {
        'query': 'Explain deep learning',
        'results': [{'score': 0.9}, {'score': 0.85}],  # Good scores
        'answer': 'Deep learning is a subset of machine learning that uses neural networks with multiple layers...',
        'latency': 2.5  # High latency
    }
]

print("=== Quality Monitoring Test ===\n")

for scenario in test_scenarios:
    result = monitor.check_quality(
        scenario['query'],
        scenario['results'],
        scenario['answer'],
        scenario['latency']
    )
    
    print(f"Query: {scenario['query']}")
    print(f"Checks passed: {result['checks_passed']}")
    print("Check details:")
    for check_name, value, passed in result['check_results']:
        status = "\u2713" if passed else "\u2717"
        print(f"  {status} {check_name}: {value}")
    print()

print("\nAlert Summary:")
print(monitor.get_alert_summary())

## 5. Production-Ready RAG with Full Monitoring

Let's combine everything into a production-ready system.

In [ ]:
class ProductionRAG:
    """
    Complete production RAG with monitoring and observability.
    """
    
    def __init__(self):
        self.rag = HybridRAG()
        self.monitor = RAGQualityMonitor()
        self.mlflow_experiment = "production_rag"
        
    def process_query(self, query):
        """Process a query with full monitoring."""
        with mlflow.start_run(run_name=f"query_{hash(query)}"):
            start_time = datetime.now()
            
            # Retrieval
            retrieval_start = datetime.now()
            results = self.rag.search(query, top_k=3)
            retrieval_latency = (datetime.now() - retrieval_start).total_seconds()
            
            # Generation
            generation_start = datetime.now()
            answer = self.rag.generate_answer(query, results)
            generation_latency = (datetime.now() - generation_start).total_seconds()
            
            total_latency = retrieval_latency + generation_latency
            
            # Quality checks
            quality_result = self.monitor.check_quality(
                query, results, answer, total_latency
            )
            
            # Log to MLflow
            mlflow.log_param("query", query[:100])
            mlflow.log_metric("retrieval_latency", retrieval_latency)
            mlflow.log_metric("generation_latency", generation_latency)
            mlflow.log_metric("total_latency", total_latency)
            mlflow.log_metric("avg_retrieval_score", 
                             sum(r.get('score', 0) for r in results) / len(results))
            mlflow.log_metric("quality_checks_passed", 
                             1 if quality_result['checks_passed'] else 0)
            
            return {
                'query': query,
                'answer': answer,
                'results': results,
                'latency': {
                    'retrieval': retrieval_latency,
                    'generation': generation_latency,
                    'total': total_latency
                },
                'quality': quality_result
            }


print("Production RAG system ready!")
print("\nTo use this system:")
print("1. Initialize: prod_rag = ProductionRAG()")
print("2. Query: result = prod_rag.process_query('your question')")
print("3. View MLflow UI: mlflow ui --port 5000")

## 6. Summary & Best Practices

### What We Covered
1. **MLflow Integration**: Experiment tracking for RAG systems
2. **Custom Visualizations**: Performance dashboards and analysis
3. **Quality Monitoring**: Automated checks and alerting
4. **Production System**: Complete monitored RAG pipeline

### Best Practices for Production RAG

1. **Always log experiments** - Track chunking, retrieval, and generation parameters
2. **Monitor key metrics** - Latency, retrieval quality, answer quality
3. **Set up alerts** - Catch quality degradation early
4. **A/B test configurations** - Use MLflow to compare different approaches
5. **Version everything** - Models, data, configurations

### Next Steps
- Deploy to production with proper MLflow server
- Add more sophisticated quality checks
- Implement feedback loops for continuous improvement